### Instalando bibliotecas

In [ ]:
import sys
!{sys.executable} -m pip install langgraph langchain langchain-google-genai langchain-community langchain-tavily langchain-chroma langchain-huggingface google-genai chromadb pandas beautifulsoup4 pydantic transformers==4.40.0 sentence-transformers==2.7.0

In [ ]:
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import sys

# 1. Remove tudo que pode estar conflitando
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio

# 2. Instala a versão específica para CUDA 11.8 (que suporta sua MX350)
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3. Garante que as bibliotecas de IA não sobrescrevam o torch
!{sys.executable} -m pip install transformers==4.40.0 sentence-transformers==2.7.0 langchain-huggingface langchain-chroma

In [1]:
import os, asyncio, json
from datetime import datetime
from getpass import getpass
from typing import TypedDict, Annotated, List, Literal
from operator import add
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_tavily import TavilySearch
from langchain.chat_models import init_chat_model
from google import genai
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.storage import LocalFileStore
from langchain_classic.storage import create_kv_docstore
from IPython.display import display, Image

import pandas as pd
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import unicodedata
import re

/home/andre/TCC/.venv/lib/python3.10/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Hugging Face Token: ")

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device}")

# O BERTimbau Base é o ideal para os 2GB de VRAM da sua MX350
model_name = "tcepi/sts_bertimbau"

# Configurações para otimizar o uso da memória de vídeo (VRAM)
model_kwargs = { 'device': device }
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 32} # Batch menor para evitar estouro de memória na MX350

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    multi_process=False
)

Dispositivo detectado: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Configurando chaves de API

In [ ]:
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
# Lista todos os modelos disponíveis para sua chave
for m in client.models.list():
    print(f"Modelo disponível: {m.name}")

Modelo disponível: models/gemini-2.5-flash
Modelo disponível: models/gemini-2.5-pro
Modelo disponível: models/gemini-2.0-flash
Modelo disponível: models/gemini-2.0-flash-001
Modelo disponível: models/gemini-2.0-flash-lite-001
Modelo disponível: models/gemini-2.0-flash-lite
Modelo disponível: models/gemini-2.5-flash-preview-tts
Modelo disponível: models/gemini-2.5-pro-preview-tts
Modelo disponível: models/gemma-4-26b-a4b-it
Modelo disponível: models/gemma-4-31b-it
Modelo disponível: models/gemini-flash-latest
Modelo disponível: models/gemini-flash-lite-latest
Modelo disponível: models/gemini-pro-latest
Modelo disponível: models/gemini-2.5-flash-lite
Modelo disponível: models/gemini-2.5-flash-image
Modelo disponível: models/gemini-3-pro-preview
Modelo disponível: models/gemini-3-flash-preview
Modelo disponível: models/gemini-3.1-pro-preview
Modelo disponível: models/gemini-3.1-pro-preview-customtools
Modelo disponível: models/gemini-3.1-flash-lite-preview
Modelo disponível: models/gemini

In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

config = {
    "system_instruction": "" 
    # Para o Gemini puro do seu teste, você pode deixar esse campo vazio ou 
    # apenas colocar uma instrução neutra para garantir que ele não se perca.
}
# Tentando o modelo 1.5 que é nativo deste novo SDK
response = client.models.generate_content(
    model="gemini-3.1-flash-lite", 
    contents="Oi.",
    config=config
)
print("✅ Conexão estabelecida!", response.text)

✅ Conexão estabelecida! Olá! Sim, estou ativo e pronto para ajudar. 😊 

Como posso te ajudar hoje?


### Construção da base de dados

In [ ]:
df = pd.read_csv("Dados/base_noticias.csv", sep=';', encoding='utf-8-sig')

print('Shape:', df.shape)

df = pd.read_csv("Dados/base_noticias.csv", sep=';', encoding='utf-8-sig')
datas_datetime = pd.to_datetime(df["data"], format="%d/%m/%Y")

mascara_abril_2026 = datas_datetime >= "2026-04-01"

links_removidos = df.loc[mascara_abril_2026, "link"].tolist()
caminho_json = "links_removidos.json"
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(links_removidos, f, ensure_ascii=False, indent=4)

df_filtrado = df[~mascara_abril_2026].reset_index(drop=True)

print('Novo shape:', df_filtrado.shape)

# 2. Lidar com valores nulos (NaN) para evitar erros na concatenação
# Preenchemos textos nulos com string vazia
colunas_texto = ['titulo', 'subtitulo', 'texto', 'data', 'link', 'verificado']
df_filtrado[colunas_texto] = df_filtrado[colunas_texto].fillna('')
df_filtrado['fontes'] = df_filtrado['fontes'].fillna('').apply(lambda x: x.split('|') if x != '' else ["Não informado"])
df_filtrado['tags'] = df_filtrado['tags'].fillna('').apply(lambda x: x.split('|') if x != '' else ["Não informado"])

fs = LocalFileStore("./docstore")
store = create_kv_docstore(fs)

documentos = []

# 3. Iterar pelas linhas do CSV construindo os Documentos do LangChain
for index, row in df_filtrado.iterrows():
    id_parent = str(index)
    conteudo_parent = f"TITULO: {row['titulo']}. | SUBTITULO: {row['subtitulo']}. | NOTICIA: {row['texto']}"
    doc_parent = Document(page_content=conteudo_parent)
    store.mset([(id_parent, doc_parent)])
    
    # CONTEÚDO SEMÂNTICO (O que o modelo vai "ler" e vetorizar)
    conteudo_para_vetorizar = (
        f"TITULO: {row['titulo']}. | "
        f"SUBTITULO: {row['subtitulo']}. | "
        f"NOTICIA: {row['texto']}"
    )
    
    # METADADOS (Dados auxiliares para filtro e citação na resposta final)
    metadados = {
        "id": id_parent,
        "link": row['link'],
        "data": row['data'],
        "titulo": row['titulo'],
        "subtitulo": row['subtitulo'],
        "fontes": row['fontes'],
        "tags": row['tags'],
        "verificado": row['verificado']
    }
    
    # Criamos o objeto Document esperado pelo LangChain/ChromaDB
    doc = Document(page_content=conteudo_para_vetorizar, metadata=metadados)
    documentos.append(doc)


# Quebramos o texto em pedaços de 1000 caracteres, com 200 caracteres de sobreposição (overlap)
# O overlap impede que uma frase importante seja cortada no meio entre dois chunks.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n", ". ", " ", ""]
)

# Aplicamos a divisão nos documentos
documentos_fragmentados = text_splitter.split_documents(documentos)

print(f"Total de documentos originais: {len(documentos)}")
print(f"Total de fragmentos (chunks) gerados: {len(documentos_fragmentados)}")

persist_directory = "./chroma_db"

# 2. Inicializa o ChromaDB VAZIO (sem documentos ainda)
# Usamos o construtor simples para definir onde o banco será salvo
vector_db = Chroma(
    collection_name="base_noticias",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

# 3. Insere o código de processamento em pedaços (Batch)
batch_size = 500 # Processa de 100 em 100 para respeitar a cota da API

print(f"Iniciando a ingestão de {len(documentos_fragmentados)} fragmentos...")

for i in range(4500, len(documentos_fragmentados), batch_size):
    batch = documentos_fragmentados[i:i + batch_size]
    
    # Adiciona o lote atual ao banco já existente
    vector_db.add_documents(batch)
    
    print(f"Progresso: {i + len(batch)}/{len(documentos_fragmentados)}")

print("Banco de dados vetorial criado e persistido com sucesso!")

Shape: (4648, 8)
Novo shape: (4602, 8)
Total de documentos originais: 4602
Total de fragmentos (chunks) gerados: 36870
Iniciando a ingestão de 36870 fragmentos...
Progresso: 500/36870
Progresso: 1000/36870
Progresso: 1500/36870
Progresso: 2000/36870
Progresso: 2500/36870
Progresso: 3000/36870
Progresso: 3500/36870
Progresso: 4000/36870
Progresso: 4500/36870
